# Library 06: figures

We draw all network pictures with Matplotlib and follow **the same rules in every format**, so the three
networks can be put side by side:

* **Node size:** strength (total partnership runs of the batter).
* **Edge width:** partnership runs of the pair.
* **Node colour:** Louvain community. Communities with fewer than 3 players are grey.
* **Labels:** only the most prominent batters, otherwise the picture becomes unreadable.
* **Positions:** a spring-style layout (Kamada-Kawai) on the plain ties. It places batters who share many
  partners close together. It has no randomness, so the pictures look the same on every run. The positions
  only help the eye; they are not a measure.
* **Readability fixes:** the big batters all sit in the middle of the layout and their circles end up on top of
  each other, so we gently push overlapping circles apart. Each label is placed above its node, or below,
  beside or diagonally next to it if that spot would cover another label or another node.

Node and edge sizes use fixed reference values, so the same size means the same number of runs in every panel.

Needs `lib_00_settings`.

In [ ]:
import matplotlib

matplotlib.use("Agg")   # draw straight to files, no window needed
import matplotlib.pyplot as plt
import matplotlib.transforms
from matplotlib.lines import Line2D

COMMUNITY_COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#6250d6"]   # blue, orange, green, violet
SMALL_COMMUNITY_COLOUR = "#9a9993"                                  # grey
TEXT_COLOUR, MUTED_TEXT_COLOUR, BACKGROUND_COLOUR = "#0b0b0b", "#52514e", "#ffffff"
EDGE_COLOUR = "#8a8a86"
SMALLEST_COLOURED_COMMUNITY = 3

# Fixed scales: the biggest strength (15,303 runs) and the heaviest edge (2,697 runs) in any of the formats.
STRENGTH_REFERENCE = 15303.0
EDGE_WEIGHT_REFERENCE = 2697.0


def node_area(strength: float, size_scale: float = 1.0) -> float:
    """Marker area for a batter with this strength. The small constant keeps low-strength batters visible."""
    return size_scale * (25 + 1100 * strength / STRENGTH_REFERENCE)


def edge_width(runs: float) -> float:
    """Line width for a pair with this many partnership runs (pairs with 0 runs still get a thin line)."""
    return 0.25 + 5.0 * runs / EDGE_WEIGHT_REFERENCE


def layout_positions(partnership_graph: nx.Graph) -> dict:
    """Kamada-Kawai positions computed on the plain ties (weight ignored).

    We leave the runs out of the layout on purpose: the heavy pairs would otherwise be pulled on top of each
    other. The runs are shown through edge width instead.
    """
    return nx.kamada_kawai_layout(partnership_graph, weight=None)


def push_overlapping_nodes_apart(positions: dict, radius_of: dict, rounds: int = 300) -> dict:
    """Move nodes apart until no two circles overlap (radius_of is in layout units).

    In each round every pair that overlaps is pushed apart along the line between them, each node by half of
    the overlap. Big hubs only move a little, so the overall shape of the layout stays the same.
    Edge case: two nodes on exactly the same spot have no direction to move in, so we push them sideways.
    """
    players = list(positions)
    coordinates = np.array([positions[player] for player in players], dtype=float)
    radii = np.array([radius_of[player] for player in players])
    for _ in range(rounds):
        moved = False
        for first in range(len(players)):
            for second in range(first + 1, len(players)):
                gap = coordinates[second] - coordinates[first]
                distance = float(np.hypot(*gap))
                needed = radii[first] + radii[second]
                if distance < needed:
                    direction = gap / distance if distance > 1e-9 else np.array([1.0, 0.0])
                    shift = direction * (needed - distance) / 2
                    coordinates[first] -= shift
                    coordinates[second] += shift
                    moved = True
        if not moved:
            break
    return {player: coordinates[index] for index, player in enumerate(players)}


def community_colours(node_measure_table: pd.DataFrame):
    """Give each node its community colour and build the legend. Community 1 is always the largest."""
    community_sizes = node_measure_table.groupby("community").size()
    colour_of_community, legend_entries, colours_used = {}, [], 0
    for community_number in sorted(community_sizes.index):
        size = community_sizes[community_number]
        # Edge case: tiny communities (1 or 2 players) and any community beyond the fourth share the grey colour.
        if size >= SMALLEST_COLOURED_COMMUNITY and colours_used < len(COMMUNITY_COLOURS):
            colour_of_community[community_number] = COMMUNITY_COLOURS[colours_used]
            legend_entries.append((f"Community {community_number} ({size} players)", COMMUNITY_COLOURS[colours_used]))
            colours_used += 1
        else:
            colour_of_community[community_number] = SMALL_COMMUNITY_COLOUR
    if SMALL_COMMUNITY_COLOUR in colour_of_community.values():
        legend_entries.append((f"Communities of < {SMALLEST_COLOURED_COMMUNITY} players", SMALL_COMMUNITY_COLOUR))
    colour_of_node = {row.player_id: colour_of_community[row.community] for row in node_measure_table.itertuples()}
    return colour_of_node, legend_entries

## Drawing one network

In [ ]:
def draw_network(axis, partnership_graph: nx.Graph, node_measure_table: pd.DataFrame, title: str,
                 labelled_batters: int = 14, size_scale: float = 1.0, font_size: float = 8.0) -> None:
    """Draw one format's network on the given Matplotlib axis."""
    colour_of_node, legend_entries = community_colours(node_measure_table)
    strength_of = node_measure_table.set_index("player_id")["strength"]
    node_list = list(partnership_graph.nodes)
    node_areas = [node_area(strength_of[player], size_scale) for player in node_list]

    # Matplotlib sizes markers in points, the layout uses its own units. The layout spans about 2 units across
    # the axis, so we convert each marker's radius into layout units and add a little padding.
    axis_width_points = axis.get_window_extent().width * 72 / axis.figure.dpi
    points_per_layout_unit = axis_width_points / 2.6
    radius_of = {player: (np.sqrt(area / np.pi) + 1.5) / points_per_layout_unit
                 for player, area in zip(node_list, node_areas)}
    positions = push_overlapping_nodes_apart(layout_positions(partnership_graph), radius_of)
    edge_widths = [edge_width(attributes["weight"]) for _, _, attributes in partnership_graph.edges(data=True)]

    nx.draw_networkx_edges(partnership_graph, positions, ax=axis, width=edge_widths, edge_color=EDGE_COLOUR, alpha=0.45)
    nx.draw_networkx_nodes(partnership_graph, positions, ax=axis, node_size=node_areas,
                           node_color=[colour_of_node[player] for player in node_list],
                           edgecolors=BACKGROUND_COLOUR, linewidths=0.9)

    # We label the batters with the highest strength plus the four with the highest betweenness, so that the
    # "bridge" players we talk about in the report can be found in the picture too.
    batters_to_label = (set(node_measure_table.nlargest(labelled_batters, "strength")["player_id"])
                        | set(node_measure_table.nlargest(4, "betweenness")["player_id"]))
    # We need the final axis limits before measuring where labels land on the page.
    axis.set_axis_off()
    axis.margins(0.08)
    axis.autoscale_view()
    renderer = axis.figure.canvas.get_renderer()

    # Every node circle in screen coordinates, so labels can avoid covering other batters.
    node_boxes = {}
    for player, area in zip(node_list, node_areas):
        centre_x, centre_y = axis.transData.transform(positions[player])
        radius_pixels = np.sqrt(area / np.pi) * axis.figure.dpi / 72
        node_boxes[player] = matplotlib.transforms.Bbox.from_extents(
            centre_x - radius_pixels, centre_y - radius_pixels, centre_x + radius_pixels, centre_y + radius_pixels)

    # Strongest batters get their label placed first, so they get the best spot (directly above the node).
    placed_label_boxes = []
    for player_id in sorted(batters_to_label, key=lambda player: (-strength_of[player], player)):
        x_position, y_position = positions[player_id]
        gap = np.sqrt(node_areas[node_list.index(player_id)] / np.pi) + 2
        diagonal = gap * 0.75
        candidate_spots = [((0, gap), "center", "bottom"), ((0, -gap), "center", "top"),
                           ((gap, 0), "left", "center"), ((-gap, 0), "right", "center"),
                           ((diagonal, diagonal), "left", "bottom"), ((-diagonal, diagonal), "right", "bottom"),
                           ((diagonal, -diagonal), "left", "top"), ((-diagonal, -diagonal), "right", "top")]
        # We score every spot: covering another label is much worse than covering another node.
        # The spot with the lowest score wins; on a tie the earlier spot in the list wins (above first).
        best_spot, best_score = None, None
        for offset, horizontal, vertical in candidate_spots:
            trial = axis.annotate(partnership_graph.nodes[player_id]["player_name"], (x_position, y_position),
                                  xytext=offset, textcoords="offset points", ha=horizontal, va=vertical,
                                  fontsize=font_size)
            trial_box = trial.get_window_extent(renderer).expanded(1.04, 1.15)
            trial.remove()
            score = (10 * sum(trial_box.overlaps(other) for other in placed_label_boxes)
                     + sum(trial_box.overlaps(box) for player, box in node_boxes.items() if player != player_id))
            if best_score is None or score < best_score:
                best_spot, best_score, best_box = (offset, horizontal, vertical), score, trial_box
            if score == 0:
                break
        offset, horizontal, vertical = best_spot
        axis.annotate(partnership_graph.nodes[player_id]["player_name"], (x_position, y_position),
                      xytext=offset, textcoords="offset points", ha=horizontal, va=vertical,
                      fontsize=font_size, color=TEXT_COLOUR, zorder=5,
                      bbox=dict(boxstyle="round,pad=0.12", fc=BACKGROUND_COLOUR, ec="none", alpha=0.8))
        placed_label_boxes.append(best_box)

    axis.set_title(title, fontsize=font_size + 3, color=TEXT_COLOUR, loc="left")
    legend_handles = [Line2D([], [], marker="o", ls="", ms=7, mfc=colour, mec=BACKGROUND_COLOUR, label=label)
                      for label, colour in legend_entries]
    # The legend goes under the network, so it never covers any nodes.
    axis.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 0.0),
                ncol=3 if len(legend_entries) > 4 else 2,
                fontsize=font_size - 1, frameon=False, labelcolor=MUTED_TEXT_COLOUR, columnspacing=1.2,
                handletextpad=0.3)

## The figures we save

In [ ]:
FIGURE_NOTE = ("Node size: strength (total partnership runs). Edge width: partnership runs of the pair. "
               "Colour: Louvain community. Positions only help the drawing.")


def save_single_network_figure(partnership_graph: nx.Graph, node_measure_table: pd.DataFrame,
                               format_name: str, output_path: Path) -> None:
    """One large picture of one format's network."""
    figure, axis = plt.subplots(figsize=(9, 8), facecolor=BACKGROUND_COLOUR)
    # The page layout has to be fixed BEFORE drawing, because label placement measures sizes on the page.
    figure.subplots_adjust(left=0.02, right=0.98, top=0.94, bottom=0.12)
    draw_network(axis, partnership_graph, node_measure_table,
                 f"England Men {format_name} batting-partnership network "
                 f"({partnership_graph.number_of_nodes()} batters, {partnership_graph.number_of_edges()} pairs)",
                 size_scale=1.6, font_size=9)
    figure.text(0.02, 0.01, FIGURE_NOTE, fontsize=7.5, color=MUTED_TEXT_COLOUR)
    figure.savefig(output_path, dpi=300, bbox_inches="tight", facecolor=BACKGROUND_COLOUR)
    plt.close(figure)


def save_comparison_figure(graphs_by_format: dict, measures_by_format: dict, output_path: Path) -> None:
    """The three networks in a 2 x 2 grid, with the size key in the fourth panel."""
    figure, axes = plt.subplots(2, 2, figsize=(9.4, 10.2), facecolor=BACKGROUND_COLOUR)
    # Fixed before drawing, for the same reason as in the single-network figure.
    figure.subplots_adjust(wspace=0.04, hspace=0.16, left=0.01, right=0.99, top=0.97, bottom=0.05)
    panels = axes.ravel()
    for axis, format_name in zip(panels, FORMATS):
        partnership_graph = graphs_by_format[format_name]
        draw_network(axis, partnership_graph, measures_by_format[format_name],
                     f"{format_name}: {partnership_graph.number_of_nodes()} batters, "
                     f"{partnership_graph.number_of_edges()} pairs",
                     labelled_batters=9, size_scale=0.75, font_size=9)

    # The fourth panel is the key: example node sizes and edge widths, drawn with the same functions.
    key_panel = panels[3]
    key_panel.set_axis_off()
    key_panel.set_xlim(0, 1)
    key_panel.set_ylim(0, 1)
    key_panel.text(0.05, 0.93, "Key (same scales in all panels)", fontsize=11, color=TEXT_COLOUR)
    for row_number, example_runs in enumerate((500, 5000, 15000)):
        y_position = 0.80 - 0.11 * row_number
        key_panel.scatter([0.12], [y_position], s=node_area(example_runs, 0.75), color=SMALL_COMMUNITY_COLOUR,
                          edgecolors=BACKGROUND_COLOUR)
        key_panel.text(0.24, y_position, f"strength {example_runs:,} runs", va="center", fontsize=9, color=MUTED_TEXT_COLOUR)
    for row_number, example_runs in enumerate((50, 500, 2500)):
        y_position = 0.44 - 0.07 * row_number
        key_panel.plot([0.05, 0.19], [y_position, y_position], lw=edge_width(example_runs),
                       color=EDGE_COLOUR, solid_capstyle="round")
        key_panel.text(0.24, y_position, f"pair total {example_runs:,} runs", va="center", fontsize=9,
                       color=MUTED_TEXT_COLOUR)
    key_panel.text(0.05, 0.17, "Colour: Louvain community, numbered by size\nwithin each format.\n"
                   "Batters who share many partners are drawn\nclose together.",
                   fontsize=9, color=MUTED_TEXT_COLOUR, va="center", linespacing=1.35)
    figure.savefig(output_path, dpi=300, bbox_inches="tight", facecolor=BACKGROUND_COLOUR)
    plt.close(figure)


def save_degree_distribution_figure(measures_by_format: dict, output_path: Path) -> None:
    """Degree distribution of each format as three small histograms with the same axes."""
    figure, axes = plt.subplots(1, 3, figsize=(10, 2.9), sharex=True, sharey=True, facecolor=BACKGROUND_COLOUR)
    histogram_bins = np.arange(0, 42, 3)   # bins of 3 partners, wide enough for the largest degree (38)
    for axis, format_name in zip(axes, FORMATS):
        degrees = measures_by_format[format_name]["degree"]
        axis.hist(degrees, bins=histogram_bins, color=COMMUNITY_COLOURS[0], edgecolor=BACKGROUND_COLOUR, linewidth=1.2)
        axis.axvline(degrees.mean(), color=TEXT_COLOUR, lw=1, ls="--")
        axis.text(degrees.mean() + 0.8, 0.93, f"mean {degrees.mean():.1f}", transform=axis.get_xaxis_transform(),
                  fontsize=8, color=MUTED_TEXT_COLOUR)
        axis.set_title(format_name, fontsize=10, loc="left", color=TEXT_COLOUR)
        axis.set_xlabel("Degree (number of partners)", fontsize=8.5, color=MUTED_TEXT_COLOUR)
        for hidden_side in ("top", "right"):
            axis.spines[hidden_side].set_visible(False)
        axis.spines["left"].set_color("#c9c8c2")
        axis.spines["bottom"].set_color("#c9c8c2")
        axis.tick_params(labelsize=8, colors=MUTED_TEXT_COLOUR, length=0)
        axis.grid(axis="y", color="#ecebe6", lw=0.8)
        axis.set_axisbelow(True)
    axes[0].set_ylabel("Number of batters", fontsize=8.5, color=MUTED_TEXT_COLOUR)
    figure.tight_layout()
    figure.savefig(output_path, dpi=300, bbox_inches="tight", facecolor=BACKGROUND_COLOUR)
    plt.close(figure)